# 07 - Locked Evaluation and Error Analysis

This notebook evaluates a frozen bundle once on the official labelled Fashionpedia validation split. It does not tune thresholds or models and never renders an unapproved image. A smoke limit validates mechanics; final report numbers require the full desktop bundle and the complete split.

In [ ]:
import json,sys
from pathlib import Path
import pandas as pd
cwd=Path.cwd().resolve(); REPO=next((p for p in [cwd,*cwd.parents] if (p/'implementation'/'src').exists()),None)
if REPO is None: raise RuntimeError('Start Jupyter inside the repository.')
sys.path.insert(0,str(REPO/'implementation'/'src'))
from ipcv_attire.dataset_policy import load_policy
from ipcv_attire.evaluation import evaluate_locked_test
from ipcv_attire.pipeline import AttirePipeline
DATA=REPO/'implementation'/'data'; FULL=REPO/'implementation'/'models'/'classical-attire-full'; SMOKE=REPO/'implementation'/'models'/'classical-attire-smoke'
BUNDLE=FULL if (FULL/'manifest.json').exists() else SMOKE
MAXIMUM_IMAGES=None if BUNDLE==FULL else 5
if not (BUNDLE/'manifest.json').exists(): raise FileNotFoundError('Run stage 06 first.')


## Metrics and baselines

Detection: precision, recall and AP50 against boxes; centre-window is the conceptual localization baseline. Segmentation: IoU/Dice against Fashionpedia masks plus rectangular-box-mask IoU. Recognition: per-target F1, balanced accuracy and coverage; report comparisons include majority and HOG-only baselines after the full desktop run. Compliance: macro-F1, balanced accuracy, decided coverage, review rate and confusion matrix.

In [ ]:
policy=load_policy(DATA/'dataset-policy.json'); metrics=evaluate_locked_test(AttirePipeline.load(BUNDLE),image_manifest=DATA/'interim'/'manifests'/'fashionpedia-images.csv',fashionpedia_root=DATA/'raw'/'fashionpedia',validation_annotations=DATA/'raw'/'fashionpedia'/'annotations'/'instances_attributes_val2020.json',policy=policy,maximum_images=MAXIMUM_IMAGES)
print('SMOKE MECHANICS ONLY' if MAXIMUM_IMAGES else 'FULL LOCKED TEST'); print(json.dumps(metrics,indent=2))


In [ ]:
display(pd.DataFrame(metrics['recognition']).T)
display(pd.DataFrame(metrics['compliance']['confusion_matrix'],index=metrics['compliance']['confusion_matrix_labels'],columns=metrics['compliance']['confusion_matrix_labels']))


## Evidence export and interpretation

Save final full-run metrics under `implementation/outputs/metrics/`. Error figures may be exported only when their image IDs pass the same automatic risk and manual approval gate. Conclusions are limited to Fashionpedia fashion imagery; the system has not been validated on real university photographs.

In [ ]:
if MAXIMUM_IMAGES is None:
    output=REPO/'implementation'/'outputs'/'metrics'/'locked-test-metrics.json'; output.parent.mkdir(parents=True,exist_ok=True); output.write_text(json.dumps(metrics,indent=2)+'\n',encoding='utf-8'); print('Saved',output)
else: print('Smoke metrics were not saved as final evidence.')
